In [1]:
import pandas as pd
df = pd.read_excel('./data/apt_seoul.xlsx',skiprows=12,thousands=",")

C:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76666 entries, 0 to 76665
Data columns (total 21 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   NO        76666 non-null  int64  
 1   시군구       76666 non-null  object 
 2   번지        76666 non-null  object 
 3   본번        76666 non-null  int64  
 4   부번        76666 non-null  int64  
 5   단지명       76666 non-null  object 
 6   전용면적(㎡)   76666 non-null  float64
 7   계약년월      76666 non-null  int64  
 8   계약일       76666 non-null  int64  
 9   거래금액(만원)  76666 non-null  int64  
 10  동         76666 non-null  object 
 11  층         76666 non-null  int64  
 12  매수자       76666 non-null  object 
 13  매도자       76666 non-null  object 
 14  건축년도      76666 non-null  int64  
 15  도로명       76666 non-null  object 
 16  해제사유발생일   76666 non-null  object 
 17  거래유형      76666 non-null  object 
 18  중개사소재지    76666 non-null  object 
 19  등기일자      76666 non-null  object 
 20  주택유형      76666 non-null  ob

In [49]:
#최고가로 거래된 아파트 정보
df.sort_values(by="거래금액(만원)").tail()
#1억이하
df.query('`거래금액(만원)` < 10000').sort_values('거래금액(만원)').tail()

,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),...,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자,주택유형
59500,59501,서울특별시 은평구 증산동,173-1,173,1,디오반,14.460,202506,12,9900,...,3,개인,개인,2011,증산로 291,-,중개거래,서울 서대문구,25.06.19,아파트
15017,15018,서울특별시 강동구 길동,457,457,0,청광플러스원큐브2차,14.495,202601,2,9900,...,13,개인,개인,2013,천호대로175길 10,-,중개거래,서울 강동구,26.01.15,아파트
37605,37606,서울특별시 구로구 구로동,107-4,107,4,비즈트위트블루,16.040,202509,17,9900,...,7,개인,개인,2012,공원로 15-12,-,중개거래,"서울 구로구, 서울 양천구",25.10.30,아파트
3959,3960,서울특별시 동대문구 장안동,373-3,373,3,현대썬앤빌601,18.350,202602,28,9950,...,9,개인,개인,2015,장한로 91,-,중개거래,서울 동대문구,26.03.16,아파트
16396,16397,서울특별시 영등포구 신길동,261-22,261,22,신길레전드힐스,13.740,202512,23,9950,...,14,개인,개인,2013,가마산로 468,-,중개거래,서울 영등포구,25.12.30,아파트


In [50]:
# 숫자칼럼만
# df.describe()
# 문자칼럼만
# df.describe(include="object")
# df.describe(include="all") #비추

### 아파트 로딩후 전처리
불편한 칼럼이름 변경(전용면적,거래금액)

1. 시군구를 구,동으로 분리저장
2. 전용면적 -> 평
3. 평 -> 평형
4. 거래취소된 행 삭제
5. 가격에 영향없는 칼럼 삭제
6. 거래가격을 억단위로 변환
7. 계약년월+계약일 -> 요일

In [51]:
# 불편한 칼럼이름 변경(전용면적,거래금액)
# df.rename 복사본으로 출력, 원본이 안 바뀜

# df = df.rename(...)
# df.reanme(.... inplace=True) 추천

df.rename( 
    columns={
        "전용면적(㎡)":"전용면적",
        "거래금액(만원)":"거래금액"
    }, inplace=True
)

#시군구를 구,동으로 분리저장
df['구'] = df['시군구'].str.split().str[1]
df['동'] = df['시군구'].str.split().str[2]

# 전용면적 -> 평
df['평'] = df['전용면적'] / 3.3

# 평 -> 평형
def to_ph(x):
    if x < 10: return '10평이하'
    if x < 20: return '10평대'
    if x < 30: return '20평대'
    if x < 40: return '30평대'
    return '40평이상'
df['평형'] = df['평'].apply(to_ph)

# 거래취소된 행 삭제(정상거래된 행)
df = df.query('해제사유발생일 == "-" ')

# 가격에 영향없는 칼럼 삭제
df.drop(columns=['NO', '번지', '본번', '부번','매수자', '매도자', '도로명', '해제사유발생일', '거래유형', '중개사소재지',
       '등기일자', '주택유형'], inplace=True)

# 거래금액을 억단위로 변환
df['거래금액'] = df['거래금액'] / 10000

# 계약년월+계약일 -> 요일
df['계약일자'] = pd.to_datetime(
    df['계약년월'].astype(str) + df['계약일'].astype(str).str.zfill(2),
    format='%Y%m%d'
)

df['계약요일'] = df['계약일자'].dt.day_name().map({
    'Monday': '월', 'Tuesday': '화', 'Wednesday': '수',
    'Thursday': '목', 'Friday': '금', 'Saturday': '토', 'Sunday': '일'
})

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 71669 entries, 0 to 76665
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   시군구     71669 non-null  object        
 1   단지명     71669 non-null  object        
 2   전용면적    71669 non-null  float64       
 3   계약년월    71669 non-null  int64         
 4   계약일     71669 non-null  int64         
 5   거래금액    71669 non-null  float64       
 6   동       71669 non-null  object        
 7   층       71669 non-null  int64         
 8   건축년도    71669 non-null  int64         
 9   구       71669 non-null  object        
 10  평       71669 non-null  float64       
 11  평형      71669 non-null  object        
 12  계약일자    71669 non-null  datetime64[ns]
 13  계약요일    71669 non-null  object        
dtypes: datetime64[ns](1), float64(3), int64(4), object(6)
memory usage: 8.2+ MB
